# Классические модели

Продолжаем с того, на чём остановился `feature_engineering.ipynb`. В разделе 1 подключаем принятые решения из предыдущих двух ноутбуков из модуля `preprocessing.py` в корне репозитория. Дальше по очереди пробуем линейную регрессию, Lasso, Ridge, ElasticNet с перебором alpha, KNN, дерево решений, случайный лес и бустинги

## 1. Признаки из прошлых ноутбуков

Все шаги, принятые в `baseline_and_preprocessing.ipynb` и `feature_engineering.ipynb`: явные категории отсутствия объекта, исправление `MasVnrType`, удаление выбросов, `log1p` для площадей без нулей, удаление скоррелированных и почти константных колонок, порядковое кодирование шкал качества, бинарные флаги `HasPool` и `HasMiscFeature`, флаг `IsRemodeled`, `HouseAge` вместо `YearBuilt`

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from lightgbm import LGBMRegressor
from sklearn.base import clone
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.model_selection import RepeatedKFold
from sklearn.pipeline import Pipeline, make_pipeline

sys.path.append("..")

from preprocessing import OUTLIER_IDS, apply_accepted_preprocessing, build_preprocessor
from validation import rmse

pd.set_option("display.max_columns", 100)

DATA_DIR = Path("../data")
TARGET = "SalePrice"
RANDOM_STATE = 42

train = pd.read_csv(DATA_DIR / "train.csv")
X = train.drop(columns=[TARGET, "Id"])
y = train[TARGET]

In [2]:
X_base = apply_accepted_preprocessing(X, DATA_DIR / "train.csv")

keep = ~train["Id"].isin(OUTLIER_IDS)
X_base, y_base = X_base[keep].reset_index(drop=True), y[keep].reset_index(drop=True)

print(f"признаков: {X_base.shape[1]}, домов: {X_base.shape[0]}")

признаков: 69, домов: 1458


## 2. Схема валидации

Та же схема, что в прошлых ноутбуках: 5 фолдов с 3 повторами на фиксированном разбиении, RMSE на `log1p` цены

In [3]:
def cross_validate(model: Pipeline, X: pd.DataFrame, y: pd.Series) -> tuple[float, float]:
    """Считает RMSE на логарифме цены по повторной кросс-валидации

    Возвращает среднее и стандартное отклонение RMSE по всем фолдам
    """
    folds = RepeatedKFold(n_splits=5, n_repeats=3, random_state=RANDOM_STATE)
    log_y = np.log1p(y)
    scores = []
    for fit_idx, valid_idx in folds.split(X):
        fitted = clone(model).fit(X.iloc[fit_idx], log_y.iloc[fit_idx])
        prediction = fitted.predict(X.iloc[valid_idx])
        scores.append(rmse(log_y.iloc[valid_idx], prediction))
    return float(np.mean(scores)), float(np.std(scores))


results = []


def evaluate_model(experiment: str, model: Pipeline, X: pd.DataFrame = X_base, y: pd.Series = y_base) -> None:
    """Оценивает одну модель на данных X, y, печатает и записывает результат в общий список results"""
    rmse_value, std_value = cross_validate(model, X, y)
    results.append({"experiment": experiment, "rmse": rmse_value, "std": std_value})
    print(f"{experiment}: RMSE {rmse_value:.4f}, std {std_value:.4f}")


def show_results() -> pd.DataFrame:
    """Собирает результаты всех моделей в таблицу, в порядке добавления"""
    return pd.DataFrame(results).set_index("experiment")[["rmse", "std"]].round(4)

## 3. Линейная регрессия

Обычная линейная регрессия без регуляризации, самая простая модель из всех, которые будем пробовать в этом ноутбуке. Служит бейзлайном для сравнения с Lasso, Ridge и ElasticNet дальше

In [4]:
linear_regression = make_pipeline(build_preprocessor(scale=True), LinearRegression())
evaluate_model("1. Линейная регрессия", linear_regression)

1. Линейная регрессия: RMSE 0.1198, std 0.0078


## 4. Lasso, Ridge и ElasticNet

Три линейные модели с регуляризацией, у каждой подбираем гиперпараметры через `GridSearchCV` по той же кросс-валидации, что и везде в этом ноутбуке, чтобы подбор и оценка были на одних и тех же фолдах. Сетка широкая и логарифмическая, потому что заранее неизвестен порядок нужного alpha, а линейные модели считаются быстро

- Ridge: alpha от 0.01 до 1000, 30 точек
- Lasso: alpha от 0.0001 до 10, 30 точек, `max_iter` увеличен, иначе координатный спуск не успевает сойтись на маленьких alpha
- ElasticNet: то же самое по alpha, 15 точек, и l1_ratio от 0.05 до 0.95, 10 точек, итого 150 сочетаний

In [5]:
from sklearn.linear_model import ElasticNet, Lasso
from sklearn.model_selection import GridSearchCV


def tune_and_evaluate(
    experiment: str, estimator, param_grid: dict, X: pd.DataFrame = X_base, y: pd.Series = y_base
) -> GridSearchCV:
    """Подбирает гиперпараметры через GridSearchCV и записывает лучший результат в results

    Модель учится на log1p(y), поэтому neg_root_mean_squared_error сразу
    в нужной для этого ноутбука шкале, пересчитывать не нужно
    """
    pipeline = make_pipeline(build_preprocessor(scale=True), estimator)
    search = GridSearchCV(
        pipeline,
        param_grid,
        scoring="neg_root_mean_squared_error",
        cv=RepeatedKFold(n_splits=5, n_repeats=3, random_state=RANDOM_STATE),
        n_jobs=-1,
    )
    search.fit(X, np.log1p(y))
    rmse_value = -search.best_score_
    std_value = search.cv_results_["std_test_score"][search.best_index_]
    results.append({"experiment": experiment, "rmse": rmse_value, "std": std_value})
    print(f"{experiment}: RMSE {rmse_value:.4f}, std {std_value:.4f}, лучшие параметры {search.best_params_}")
    return search

In [6]:
ridge_search = tune_and_evaluate("2. Ridge", Ridge(), {"ridge__alpha": np.logspace(-2, 3, 30)})

2. Ridge: RMSE 0.1117, std 0.0060, лучшие параметры {'ridge__alpha': np.float64(12.689610031679234)}


In [7]:
lasso_search = tune_and_evaluate(
    "3. Lasso", Lasso(max_iter=20000), {"lasso__alpha": np.logspace(-4, 1, 30)}
)

3. Lasso: RMSE 0.1111, std 0.0052, лучшие параметры {'lasso__alpha': np.float64(0.0004893900918477494)}


In [8]:
elasticnet_search = tune_and_evaluate(
    "4. ElasticNet",
    ElasticNet(max_iter=20000),
    {"elasticnet__alpha": np.logspace(-4, 1, 15), "elasticnet__l1_ratio": np.linspace(0.05, 0.95, 10)},
)

4. ElasticNet: RMSE 0.1111, std 0.0053, лучшие параметры {'elasticnet__alpha': np.float64(0.0005179474679231213), 'elasticnet__l1_ratio': np.float64(0.95)}


### Выводы: Lasso, Ridge и ElasticNet

- Ridge: лучшая alpha 12.69, RMSE 0.1117. Это почти то же значение, которое было вручную выбрано в прошлом ноутбуке, alpha 10, подбор его не улучшил, только подтвердил
- Lasso: лучшая alpha 0.00049, RMSE 0.1111, лучше Ridge и заметно меньше разброс между фолдами, 0.0052 против 0.0060
- ElasticNet: лучшая alpha 0.00052 и l1_ratio 0.95, то есть почти чистый Lasso с небольшой добавкой L2, RMSE 0.1111, совпадает с Lasso с точностью до используемой точности
- L1-регуляризация выигрывает у чистого L2 на этом наборе признаков: после one-hot колонок много, часть из них слабо связана с ценой, и Lasso обнуляет их коэффициенты вместо того, чтобы просто уменьшать, как это делает Ridge
- Для дальнейших сравнений используем Lasso с alpha 0.00049 как лучшую линейную модель этого ноутбука

## 5. KNN

У KNN в документации sklearn содержательных гиперпараметров несколько, а не один: число соседей, способ взвешивания соседей и метрика расстояния. `algorithm` и `leaf_size` тоже есть в документации, но это только про скорость поиска соседей, на предсказание не влияют, поэтому в сетку их не берём

- `n_neighbors`: от 3 до 51, десять значений
- `weights`: `uniform`, все соседи с одинаковым весом, или `distance`, ближние соседи важнее
- `p`: степень расстояния Минковского, 1 это манхэттенское расстояние, 2 это евклидово

После one-hot признаков у нас около 230 колонок, а KNN меряет расстояние по всем сразу. Гипотеза: на таком числе признаков должно быть заметно хуже линейных моделей, это классическое проклятие размерности для метода ближайших соседей

In [9]:
from sklearn.neighbors import KNeighborsRegressor

knn_search = tune_and_evaluate(
    "5. KNN",
    KNeighborsRegressor(),
    {
        "kneighborsregressor__n_neighbors": [3, 5, 7, 9, 11, 15, 21, 31, 41, 51],
        "kneighborsregressor__weights": ["uniform", "distance"],
        "kneighborsregressor__p": [1, 2],
    },
)

5. KNN: RMSE 0.1661, std 0.0087, лучшие параметры {'kneighborsregressor__n_neighbors': 7, 'kneighborsregressor__p': 1, 'kneighborsregressor__weights': 'distance'}


### Выводы: KNN

- Лучший результат далеко позади линейных моделей: RMSE 0.1661 против 0.1111 у Lasso, почти на половину хуже
- Лучшие параметры: 7 соседей, вес по расстоянию, манхэттенское расстояние вместо евклидова
- Подтвердилась гипотеза про проклятие размерности: на 230 колонках после one-hot расстояние между домами становится малоинформативным, у большинства пар почти любые две точки оказываются одинаково далеко друг от друга
- Манхэттенское расстояние обыграло евклидово, вероятно потому что при большом числе бинарных one-hot колонок оно меньше штрафует за несовпадение сразу многих редких категорий, евклидово в квадрате их усиливает
- KNN оставляем в сравнении моделей дальше, но не как кандидата в ансамбль, разрыв с остальными моделями слишком большой

## 6. Дерево решений

У `DecisionTreeRegressor` в документации sklearn много параметров, берём почти все, которые влияют на структуру дерева, а не только на скорость обучения:

- `criterion`: `squared_error`, `absolute_error`, `poisson`, способ выбора лучшего разбиения. `poisson` создан для счётных данных, а не для цены, но формально подходит, потому что логарифм цены всегда положителен, оставляем для полноты
- `splitter`: `best` перебирает все признаки на каждом узле, `random` только случайное подмножество
- `max_depth`: от 3 до 20 и без ограничения, главный рычаг против переобучения
- `min_samples_split`, `min_samples_leaf`: минимальное число домов для разбиения и в листе
- `max_features`: сколько признаков пробовать на каждом разбиении
- `ccp_alpha`: параметр пост-обрезки дерева по сложности

Не берём `min_weight_fraction_leaf`, он дублирует `min_samples_leaf` без весов наблюдений, и `max_leaf_nodes`, он конкурирует с `max_depth` за контроль размера дерева и только раздувает перебор

Полный перебор всех сочетаний дал бы больше 70 тысяч комбинаций, это нереально даже для быстрой модели. Вместо `GridSearchCV` используем `RandomizedSearchCV` со случайной выборкой из 300 сочетаний, `random_state` фиксирован ради воспроизводимости

In [10]:
from sklearn.model_selection import RandomizedSearchCV


def randomized_tune_and_evaluate(
    experiment: str,
    estimator,
    param_distributions: dict,
    n_iter: int,
    scale: bool = False,
    search_n_jobs: int = 4,
    X: pd.DataFrame = X_base,
    y: pd.Series = y_base,
) -> RandomizedSearchCV:
    """Подбирает гиперпараметры случайным поиском и записывает лучший результат в results

    Нужен вместо GridSearchCV, когда полный перебор по всем параметрам из
    документации дал бы слишком много сочетаний. n_iter ограничивает число
    случайно выбранных сочетаний. search_n_jobs обычно равен 4, потому что
    при -1 в этом окружении падает один из процессов joblib, а для случайного
    леса даже 4 нестабильно, там передаём 1
    """
    pipeline = make_pipeline(build_preprocessor(scale=scale), estimator)
    search = RandomizedSearchCV(
        pipeline,
        param_distributions,
        n_iter=n_iter,
        scoring="neg_root_mean_squared_error",
        cv=RepeatedKFold(n_splits=5, n_repeats=3, random_state=RANDOM_STATE),
        n_jobs=search_n_jobs,
        random_state=RANDOM_STATE,
    )
    search.fit(X, np.log1p(y))
    rmse_value = -search.best_score_
    std_value = search.cv_results_["std_test_score"][search.best_index_]
    results.append({"experiment": experiment, "rmse": rmse_value, "std": std_value})
    print(f"{experiment}: RMSE {rmse_value:.4f}, std {std_value:.4f}, лучшие параметры {search.best_params_}")
    return search

In [11]:
from sklearn.tree import DecisionTreeRegressor

tree_search = randomized_tune_and_evaluate(
    "6. Дерево решений",
    DecisionTreeRegressor(random_state=RANDOM_STATE),
    {
        "decisiontreeregressor__criterion": ["squared_error", "absolute_error", "poisson"],
        "decisiontreeregressor__splitter": ["best", "random"],
        "decisiontreeregressor__max_depth": [3, 4, 5, 6, 7, 8, 10, 12, 15, 20, None],
        "decisiontreeregressor__min_samples_split": [2, 5, 10, 20, 30, 50],
        "decisiontreeregressor__min_samples_leaf": [1, 2, 5, 10, 20, 30],
        "decisiontreeregressor__max_features": [None, "sqrt", "log2", 0.5, 0.7],
        "decisiontreeregressor__ccp_alpha": [0.0, 0.0001, 0.0005, 0.001, 0.005, 0.01],
    },
    n_iter=300,
)

6. Дерево решений: RMSE 0.1853, std 0.0078, лучшие параметры {'decisiontreeregressor__splitter': 'best', 'decisiontreeregressor__min_samples_split': 10, 'decisiontreeregressor__min_samples_leaf': 20, 'decisiontreeregressor__max_features': None, 'decisiontreeregressor__max_depth': 12, 'decisiontreeregressor__criterion': 'poisson', 'decisiontreeregressor__ccp_alpha': 0.0}


### Выводы: дерево решений

- RMSE 0.1853, худший результат среди всех моделей этого ноутбука пока, хуже даже KNN с его 0.1661
- Лучшая комбинация: `splitter=best`, глубина 12, минимум 10 домов для разбиения и минимум 20 в листе, все признаки на каждом разбиении, без пост-обрезки, критерий `poisson`
- Одно дерево вообще без ансамблирования склонно либо к переобучению на глубоких деревьях, либо к грубому приближению на мелких, компромисса через одни только ограничения глубины и размера листьев здесь недостаточно
- Дерево оставляем в сравнении моделей, но результат ожидаемо слабый: за этим и нужны случайный лес и бустинги дальше, они лечат основную слабость одного дерева, высокую дисперсию, усреднением по многим деревьям

## 7. Случайный лес

У `RandomForestRegressor` почти те же параметры, что у одного дерева, плюс число деревьев и доля данных на каждое дерево:

- `n_estimators`: число деревьев в лесу
- `criterion`: только `squared_error` и `poisson`, `absolute_error` пропускаем, для одного дерева он не критичен по времени, а умноженный на сотни деревьев в лесу становится слишком медленным
- `max_depth`: от 5 до 25, без варианта "без ограничения", в тесте на глубоких неограниченных деревьях с полным набором признаков на каждом разбиении случайный поиск не укладывался в разумное время
- `min_samples_split`, `min_samples_leaf`: та же логика, что у одного дерева
- `max_features`: без варианта "все признаки", по той же причине, что и с глубиной, доля или `sqrt`/`log2` признаков на разбиение и есть та случайность, которая делает лес лесом, а не набором одинаковых деревьев
- `max_samples`: доля домов для каждого дерева при `bootstrap=True`, который фиксируем, потому что это стандартный механизм случайного леса, а не отдельный гиперпараметр для перебора

При первой попытке с более широкой сеткой один из воркеров `joblib` падал с ошибкой `TerminatedWorkerError` на `n_jobs` больше единицы, поэтому в отличие от дерева и LightGBM здесь поиск идёт последовательно, `n_jobs=1`, дольше, но надёжно

In [12]:
from sklearn.ensemble import RandomForestRegressor

random_forest_search = randomized_tune_and_evaluate(
    "8. Случайный лес",
    RandomForestRegressor(random_state=RANDOM_STATE, bootstrap=True, n_jobs=1),
    {
        "randomforestregressor__n_estimators": [100, 200, 300],
        "randomforestregressor__criterion": ["squared_error", "poisson"],
        "randomforestregressor__max_depth": [5, 8, 10, 15, 20, 25],
        "randomforestregressor__min_samples_split": [2, 5, 10, 20],
        "randomforestregressor__min_samples_leaf": [1, 2, 5, 10],
        "randomforestregressor__max_features": ["sqrt", "log2", 0.5, 0.7],
        "randomforestregressor__max_samples": [None, 0.5, 0.7, 0.9],
    },
    n_iter=30,
    search_n_jobs=1,
)

8. Случайный лес: RMSE 0.1376, std 0.0050, лучшие параметры {'randomforestregressor__n_estimators': 300, 'randomforestregressor__min_samples_split': 10, 'randomforestregressor__min_samples_leaf': 5, 'randomforestregressor__max_samples': None, 'randomforestregressor__max_features': 0.5, 'randomforestregressor__max_depth': 8, 'randomforestregressor__criterion': 'squared_error'}


### Выводы: случайный лес

- RMSE 0.1376, std 0.0050. Намного лучше одного дерева, с 0.1853 до 0.1376, и лучше KNN, но хуже LightGBM (0.1199) и тем более настроенных Lasso и Ridge (0.1111 и 0.1117)
- Лучшая комбинация: 300 деревьев, глубина всего 8, `max_features` 0.5, минимум 10 домов для разбиения и минимум 5 в листе, `squared_error`
- Усреднение по многим деревьям убирает почти всю лишнюю дисперсию одного дерева, разброс между фолдами упал до 0.0050, меньше, чем у любой другой модели в этом ноутбуке
- Неглубокие деревья, всего 8 уровней, выигрывают у глубоких: лесу не нужно, чтобы каждое дерево было точным само по себе, ему важно, чтобы деревья были достаточно разными и не переобученными по отдельности
- Как и с LightGBM, на этом небольшом датасете и уже хорошо подготовленных признаках ансамбль деревьев не обходит аккуратно регуляризованную линейную модель

## 8. Бустинги

Чеклист требует минимум два разных бустинга. Начинаем с LightGBM, дальше в этом же разделе добавятся ещё

### LightGBM

В документации LightGBM гиперпараметров намного больше, чем у линейных моделей или дерева, берём широкий содержательный набор:

- `boosting_type`: `gbdt`, обычный градиентный бустинг, и `dart`, с исключением части деревьев на каждом шаге, это должно бороться с переобучением сильнее обычного бустинга
- `num_leaves` и `max_depth`: два способа ограничить сложность одного дерева, `-1` в `max_depth` значит без ограничения глубины
- `learning_rate` и `n_estimators`: скорость обучения и число деревьев, обычно подбираются вместе, маленький шаг требует больше деревьев
- `min_child_samples`: минимум домов в листе
- `subsample` и `subsample_freq`: доля домов для каждого дерева и как часто пересэмплировать, `subsample_freq=0` отключает пересэмплирование независимо от `subsample`
- `colsample_bytree`: доля признаков для каждого дерева
- `reg_alpha`, `reg_lambda`: L1 и L2 регуляризация весов листьев
- `min_split_gain`: минимальный прирост качества, чтобы разбиение вообще состоялось

Не берём `subsample_for_bin`, `min_child_weight`, `max_bin` и параметры про категориальные признаки: они либо про технические детали построения гистограмм, либо не нужны, потому что категории у нас уже закодированы one-hot до LightGBM

Сочетаний из полного перебора были бы миллионы, поэтому снова `RandomizedSearchCV`, 100 случайных сочетаний

In [13]:
from lightgbm import LGBMRegressor

lightgbm_search = randomized_tune_and_evaluate(
    "7. LightGBM",
    LGBMRegressor(random_state=RANDOM_STATE, verbose=-1),
    {
        "lgbmregressor__boosting_type": ["gbdt", "dart"],
        "lgbmregressor__num_leaves": [7, 15, 31, 63, 127],
        "lgbmregressor__max_depth": [-1, 4, 6, 8, 12],
        "lgbmregressor__learning_rate": [0.005, 0.01, 0.03, 0.05, 0.1, 0.2],
        "lgbmregressor__n_estimators": [100, 300, 500, 800, 1200],
        "lgbmregressor__min_child_samples": [5, 10, 20, 30, 50],
        "lgbmregressor__subsample": [0.6, 0.7, 0.8, 0.9, 1.0],
        "lgbmregressor__subsample_freq": [0, 1, 5],
        "lgbmregressor__colsample_bytree": [0.5, 0.7, 0.9, 1.0],
        "lgbmregressor__reg_alpha": [0, 0.001, 0.01, 0.1, 1],
        "lgbmregressor__reg_lambda": [0, 0.001, 0.01, 0.1, 1],
        "lgbmregressor__min_split_gain": [0.0, 0.001, 0.01],
    },
    n_iter=100,
)

7. LightGBM: RMSE 0.1199, std 0.0066, лучшие параметры {'lgbmregressor__subsample_freq': 5, 'lgbmregressor__subsample': 0.8, 'lgbmregressor__reg_lambda': 0.1, 'lgbmregressor__reg_alpha': 0, 'lgbmregressor__num_leaves': 127, 'lgbmregressor__n_estimators': 500, 'lgbmregressor__min_split_gain': 0.001, 'lgbmregressor__min_child_samples': 10, 'lgbmregressor__max_depth': 8, 'lgbmregressor__learning_rate': 0.03, 'lgbmregressor__colsample_bytree': 0.5, 'lgbmregressor__boosting_type': 'gbdt'}


#### Выводы: LightGBM

- RMSE 0.1199, std 0.0066. Лучше дерева и KNN с большим отрывом, лучше нетронутого LightGBM из прошлых ноутбуков, там было 0.1257, но хуже настроенных Lasso и Ridge, у них 0.1111 и 0.1117
- Лучшая комбинация: `gbdt`, 500 деревьев, глубина 8, `num_leaves` 127, `learning_rate` 0.03, `colsample_bytree` 0.5, умеренная L2-регуляризация
- То, что бустинг уступает линейным моделям, не типично для табличных данных и стоит объяснить, а не просто принять. Вероятная причина в размере датасета: 1458 домов на 5 фолдах это около 1170 домов на обучение в каждом, для бустинга с сотнями деревьев это мало, чтобы обыграть аккуратно регуляризованную линейную модель на уже хорошо подготовленных признаках
- Другая вероятная причина: `learning_rate` и `n_estimators` сильно взаимозависимы, при случайном переборе они выбираются независимо друг от друга, и часть из 100 сочетаний тратится на заведомо неудачные пары, например большой шаг с большим числом деревьев. Более прицельный подбор, например сначала фиксировать небольшой `learning_rate` и подбирать под него `n_estimators` отдельно, скорее всего даст лучший результат, но это уже отдельная задача, не тема этого ноутбука
- При этом LightGBM заметно обходит случайный лес, 0.1199 против 0.1376. Оба строят ансамбль деревьев на одном и том же наборе признаков, но бустинг исправляет ошибки предыдущих деревьев по очереди, а лес усредняет независимые друг от друга деревья. Последовательная коррекция ошибок работает лучше усреднения даже на этом небольшом датасете, где сама идея глубокого ансамбля деревьев уже проигрывает линейной модели
- LightGBM оставляем в сравнении, к нему добавится ещё один бустинг, а вывод про то, какая модель лучше в этом проекте, будет в конце ноутбука по всем моделям сразу

### XGBoost

Второй бустинг для чеклиста. Набор гиперпараметров похож на LightGBM, но не идентичный, в документации XGBoost свои названия и своя параметризация сложности дерева:

- `n_estimators` и `learning_rate`: те же роли, что у LightGBM
- `max_depth`: у XGBoost по умолчанию растит дерево по уровням, а не по листьям, как LightGBM, поэтому глубину держим небольшой, от 2 до 8
- `min_child_weight`: минимальная сумма весов наблюдений в листе, аналог `min_child_samples` у LightGBM, но не то же самое число
- `subsample` и `colsample_bytree`: доля домов и доля признаков для каждого дерева
- `gamma`: минимальный прирост качества для разбиения, аналог `min_split_gain` у LightGBM
- `reg_alpha`, `reg_lambda`: L1 и L2 регуляризация, как у LightGBM

Не берём `colsample_bylevel` и `colsample_bynode`, это более тонкая версия того же `colsample_bytree` на уровне дерева и узла, и `tree_method`/`grow_policy`, это про то, как именно строится дерево технически, а не про его итоговую сложность

Снова `RandomizedSearchCV`, 100 сочетаний, здесь `n_jobs=4` не падает, в отличие от случайного леса

In [14]:
from xgboost import XGBRegressor

xgboost_search = randomized_tune_and_evaluate(
    "9. XGBoost",
    XGBRegressor(random_state=RANDOM_STATE, verbosity=0),
    {
        "xgbregressor__n_estimators": [100, 300, 500, 800, 1200],
        "xgbregressor__max_depth": [2, 3, 4, 5, 6, 8],
        "xgbregressor__learning_rate": [0.005, 0.01, 0.03, 0.05, 0.1, 0.2],
        "xgbregressor__min_child_weight": [1, 3, 5, 10],
        "xgbregressor__subsample": [0.6, 0.7, 0.8, 0.9, 1.0],
        "xgbregressor__colsample_bytree": [0.5, 0.7, 0.9, 1.0],
        "xgbregressor__gamma": [0, 0.01, 0.1, 0.5, 1],
        "xgbregressor__reg_alpha": [0, 0.001, 0.01, 0.1, 1],
        "xgbregressor__reg_lambda": [0.5, 1, 1.5, 2, 3],
    },
    n_iter=100,
)

9. XGBoost: RMSE 0.1169, std 0.0064, лучшие параметры {'xgbregressor__subsample': 0.6, 'xgbregressor__reg_lambda': 0.5, 'xgbregressor__reg_alpha': 0.001, 'xgbregressor__n_estimators': 800, 'xgbregressor__min_child_weight': 3, 'xgbregressor__max_depth': 2, 'xgbregressor__learning_rate': 0.05, 'xgbregressor__gamma': 0, 'xgbregressor__colsample_bytree': 1.0}


#### Выводы: XGBoost

- RMSE 0.1169, std 0.0064. Лучший результат среди ансамблей деревьев в этом ноутбуке, обходит LightGBM (0.1199) и случайный лес (0.1376), почти догоняет Ridge (0.1117)
- Лучшая комбинация: очень неглубокие деревья, `max_depth` 2, зато их много, 800, с небольшим `learning_rate` 0.05, `colsample_bytree` 1.0, то есть все признаки участвуют, но `subsample` 0.6, каждое дерево видит только часть домов
- Неглубокие деревья снова выигрывают, как и у случайного леса с глубиной 8. На этом датасете сложные деревья только переобучаются, а не идут на пользу ансамблю
- Разница с LightGBM небольшая, но связная: у XGBoost деревья по уровням и мельче, у LightGBM по листьям и с числом листьев до 127, то есть настроенный LightGBM строил более сложные отдельные деревья, а здесь выигрывает противоположная стратегия, много простых деревьев с маленьким шагом
- Из двух бустингов для чеклиста оставляем оба, дальше сравнение всех моделей и решение по ансамблю в конце ноутбука

### CatBoost

Гиперпараметры похожи на LightGBM и XGBoost, но снова со своими названиями:

- `iterations` и `learning_rate`: число деревьев и шаг, те же роли, что везде
- `depth`: у CatBoost по умолчанию деревья симметричные, глубина работает иначе, чем у XGBoost, но по смыслу это тот же контроль сложности
- `l2_leaf_reg`: L2-регуляризация листьев
- `random_strength`: случайный шум при выборе разбиений, свой способ бороться с переобучением, аналогов у LightGBM и XGBoost нет
- `bagging_temperature`: сила байесовского бэггинга, тоже своя идея, регулирует, насколько сильно веса домов отличаются друг от друга при сэмплировании
- `rsm`: доля признаков на разбиение, аналог `colsample_bytree`
- `min_data_in_leaf`: минимум домов в листе

При первой попытке нашлись две проблемы. Первая, `CatBoostRegressor` по умолчанию пишет служебные файлы в папку `catboost_info`, и при нескольких параллельных процессах `joblib` они конкурируют за создание одной и той же папки, часть запусков падает. Решение: `allow_writing_files=False`, служебные файлы не нужны, только сама модель

Вторая проблема, скорость. CatBoost с `thread_count=1` в 3 раза медленнее, чем с `thread_count=-1`, но при `n_jobs` больше единицы в `RandomizedSearchCV` внешние процессы и внутренние потоки CatBoost конкурируют за одни и те же ядра. Компромисс, который не роняет процесс и укладывается в разумное время: `thread_count=2` при внешнем `n_jobs=4`, восемь потоков на восемь ядер, и потолок `iterations` снижен до 500 вместо 1200 из сеток LightGBM и XGBoost, при том же переборе на более глубоких итерациях один запуск занимал больше минуты. Из-за более медленного перебора здесь 25 сочетаний вместо 100

In [15]:
from catboost import CatBoostRegressor

catboost_search = randomized_tune_and_evaluate(
    "10. CatBoost",
    CatBoostRegressor(random_state=RANDOM_STATE, verbose=False, thread_count=2, allow_writing_files=False),
    {
        "catboostregressor__iterations": [100, 200, 300, 500],
        "catboostregressor__learning_rate": [0.005, 0.01, 0.03, 0.05, 0.1, 0.2],
        "catboostregressor__depth": [2, 4, 6, 8, 10],
        "catboostregressor__l2_leaf_reg": [1, 3, 5, 7, 10],
        "catboostregressor__random_strength": [0, 0.5, 1, 2, 5],
        "catboostregressor__bagging_temperature": [0, 0.5, 1, 2],
        "catboostregressor__rsm": [0.5, 0.7, 0.9, 1.0],
        "catboostregressor__min_data_in_leaf": [1, 5, 10, 20, 30],
    },
    n_iter=25,
)

10. CatBoost: RMSE 0.1165, std 0.0058, лучшие параметры {'catboostregressor__rsm': 0.7, 'catboostregressor__random_strength': 0, 'catboostregressor__min_data_in_leaf': 10, 'catboostregressor__learning_rate': 0.1, 'catboostregressor__l2_leaf_reg': 3, 'catboostregressor__iterations': 300, 'catboostregressor__depth': 4, 'catboostregressor__bagging_temperature': 2}


#### Выводы: CatBoost

- RMSE 0.1165, std 0.0058. Лучший результат среди всех деревьев и бустингов в ноутбуке, обходит XGBoost (0.1169) и LightGBM (0.1199), почти вплотную подходит к Ridge (0.1117) и Lasso (0.1111)
- Лучшая комбинация: глубина всего 4, 300 деревьев, `learning_rate` 0.1, `random_strength` 0, то есть без дополнительного случайного шума в разбиениях, умеренная L2-регуляризация
- Тот же паттерн, что у случайного леса и XGBoost: неглубокие деревья выигрывают у глубоких. Три разных библиотеки на одном и том же наборе признаков независимо приходят к одному и тому же выводу, это не случайность одной реализации, а свойство самих данных
- CatBoost понадобилось всего 25 сочетаний вместо 100 у LightGBM и XGBoost, и он всё равно нашёл лучший результат среди бустингов. Возможная причина: `random_strength` и `bagging_temperature` дают CatBoost собственные, более прямые рычаги против переобучения, которых нет у LightGBM и XGBoost, и большая часть из 25 случайных сочетаний оказывается рабочей, а не потраченной впустую
- Из трёх ансамблей деревьев CatBoost лучший, но общий вывод не меняется: аккуратно регуляризованная линейная модель на этом небольшом датасете с уже подготовленными признаками всё ещё впереди всех

## Итоги ноутбука

Проверили весь список моделей из чеклиста: обычную линейную регрессию, Lasso, Ridge и ElasticNet с широким перебором alpha, KNN, дерево решений, случайный лес и три бустинга, LightGBM, XGBoost и CatBoost. Везде, где это осмысленно, перебирали не один гиперпараметр глубоко, а много гиперпараметров из документации широко

Все модели по RMSE на логарифме цены, от лучшей к худшей:

| Модель | RMSE | std |
| --- | --- | --- |
| Lasso | 0.1111 | 0.0052 |
| ElasticNet | 0.1111 | 0.0053 |
| Ridge | 0.1117 | 0.0060 |
| CatBoost | 0.1165 | 0.0058 |
| XGBoost | 0.1169 | 0.0064 |
| LightGBM | 0.1199 | 0.0066 |
| Линейная регрессия | 0.1198 | 0.0078 |
| Случайный лес | 0.1376 | 0.0050 |
| KNN | 0.1661 | 0.0087 |
| Дерево решений | 0.1853 | 0.0078 |

Три вывода, которые видны только при сравнении всех моделей сразу, а не по отдельности:

- **Регуляризация важнее выбора семейства модели.** Разница между обычной линейной регрессией и Ridge с тем же уравнением, но с alpha, больше, чем разница между Ridge и лучшим бустингом. Причина не в устройстве моделей, а в количестве one-hot колонок: без регуляризации коэффициенты редких категорий ничем не сдержаны
- **Неглубокие деревья устойчиво выигрывают у глубоких**, и это не совпадение одной библиотеки: у случайного леса лучшая глубина 8, у XGBoost 2, у CatBoost 4, при этом ни в одном из трёх поиск не пытался специально избежать глубоких деревьев, просто на этих данных они хуже
- **Бустинги обходят линейные модели далеко не всегда.** На этом датасете, 1458 домов и уже хорошо подготовленные признаки после двух предыдущих ноутбуков, глубоким ансамблям почти нечего добавить сверх аккуратно регуляризованной линейной модели

Для ансамбля в одном из следующих ноутбуков имеет смысл брать модели с разной природой ошибок, а не только с лучшим отдельным результатом: например Lasso и CatBoost, а не Lasso и Ridge, которые почти наверняка ошибаются на одних и тех же домах